In [1]:
!pip install langchain langchain-community langchain-groq faiss-cpu sentence-transformers PyPDF2 pypdf langchain-huggingface --q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install fpdf --q

  Preparing metadata (setup.py) ... done


In [4]:
from fpdf import FPDF
import os
pdf = FPDF()
pdf.add_page()
pdf.set_font("Arial",size=14)
content = [
    "AI is designed to simulate human intelligence in machines.",
    "Machine learning is a subset of AI focused on systems learning from data.",
    "Deep learning involves neural networks with many layers to learn complex patterns.",
    "Natural Language Processing (NLP) enables computers to understand human language.",
    "Computer Vision allows machines to interpret and understand visual information.",
    "Robotics combines AI with engineering to create intelligent machines.",
    "AI is used in recommendation systems to suggest personalized content.",
    "Autonomous vehicles rely heavily on AI for navigation and decision-making.",
    "Generative AI creates new content like text, images, and music.",
    "AI assists in data analysis to uncover insights and make predictions."
]
for line in content:
  pdf.multi_cell(0,20,line)

pdf_file= "AI.pdf"
pdf.output(pdf_file)
print(f"pdf saved as {pdf_file}")
os.listdir(".")

pdf saved as AI.pdf


['.config', 'AI.pdf', 'sample_data']

In [10]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings # Updated import
from langchain_community.vectorstores import faiss
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"]="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"

loader = PyPDFLoader("AI.pdf")
pages = loader.load()

splitter = CharacterTextSplitter(chunk_size =500,chunk_overlap=50)
documents = splitter.split_documents(pages)

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = faiss.FAISS.from_documents(documents,embedding)

llm = ChatGroq(model="llama-3.1-8b-instant")
prompt = PromptTemplate.from_template(
    "Use the following context to answer the question:\n\ncontext:\n{context}\n\nQuestion:\n{question}\n Answer:"
)
parser = StrOutputParser()
def ask_pdf(query:str):
  docs= db.similarity_search_with_score(query,k=3)
  context = "\n\n".join([d[0].page_content for d in docs])
  chain= prompt|llm|parser
  return chain.invoke({"context":context,"question":query})

while True:
  query=input("\nAsk the question:")
  if query.lower()=="exit":
    break
  answer = ask_pdf(query)
  print(answer)

/tmp/ipykernel_7300/754661931.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Ask the question:give me some important notes from this
Here are some important notes from the context:

1. **Subsets of AI:** 
   - Machine learning: focuses on systems learning from data.
   - Deep learning: involves neural networks with many layers to learn complex patterns.

2. **Applications of AI:**
   - Natural Language Processing (NLP): enables computers to understand human language.
   - Computer Vision: allows machines to interpret and understand visual information.
   - Robotics: combines AI with engineering to create intelligent machines.
   - Recommendation systems: uses AI to suggest personalized content.
   - Autonomous vehicles: relies heavily on AI for navigation and decision-making.
   - Generative AI: creates new content like text, images, and music.
   - Data analysis: AI assists in data analysis to uncover insights and make predictions.

3. **Key areas where AI is applied:**
   - Human language (NLP)
   - Visual information (Computer Vision)
   - Personalized cont